# Appendix — Radiomics pipeline demo (nii.gz → HU clip → ring → features)

Mục đích của notebook này:
- Hiểu luồng xử lý ảnh CT ở mức demo.
- Không dùng dữ liệu bệnh nhân thật.
- Tạo 1 volume CT giả lập và 1 mask giả lập.
- Làm HU clip, tạo ring 1/3/5 mm, tính vài feature đơn giản.

Nếu không có thời gian, có thể đọc phần markdown và chạy vài cell đầu.

In [ ]:
!pip -q install nibabel scipy


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
from scipy.ndimage import distance_transform_edt

plt.rcParams['figure.figsize'] = (6,4)


## 1) Tạo CT giả lập và mask giả lập
CT giả lập có HU trong khoảng -1000 đến 200.
Mask giả lập là một khối u dạng cầu ở giữa volume.

In [ ]:
# volume kích thước nhỏ để chạy nhanh
shape = (96, 96, 96)
spacing = (1.0, 1.0, 1.0)  # mm

rng = np.random.default_rng(42)
ct = rng.normal(loc=-600, scale=120, size=shape)  # nền phổi
ct = np.clip(ct, -1000, 200)

# tạo khối u dạng cầu
zz, yy, xx = np.meshgrid(
    np.arange(shape[0]), np.arange(shape[1]), np.arange(shape[2]),
    indexing="ij"
)
center = np.array(shape) / 2
radius = 18
dist = np.sqrt(((zz-center[0])**2 + (yy-center[1])**2 + (xx-center[2])**2))
tumor_mask = (dist <= radius).astype(np.uint8)

ct[tumor_mask==1] = rng.normal(loc=-200, scale=80, size=int(tumor_mask.sum()))

ct.shape, tumor_mask.sum()


In [ ]:
# xem 1 lát cắt
z = shape[0]//2
plt.imshow(ct[z,:,:], cmap="gray")
plt.contour(tumor_mask[z,:,:], colors="r", linewidths=1)
plt.title("CT demo + tumor mask (slice)")
plt.axis("off")
plt.show()


## 2) HU clip
Trong dự án thường clip HU để giảm outlier và chuẩn hoá.

In [ ]:
ct_clip = np.clip(ct, -800, 800)


## 3) Tạo ring theo mm
Dùng distance map tính khoảng cách từ bờ u ra ngoài.
Ring1: 0–1 mm ngoài u. Ring3: 0–3 mm. Ring5: 0–5 mm.

In [ ]:
# distance outside tumor: tính khoảng cách từ voxel ngoài u tới bờ u (mm)
outside = 1 - tumor_mask
dist_outside_mm = distance_transform_edt(outside, sampling=spacing)

ring1 = ((dist_outside_mm > 0) & (dist_outside_mm <= 1)).astype(np.uint8)
ring3 = ((dist_outside_mm > 0) & (dist_outside_mm <= 3)).astype(np.uint8)
ring5 = ((dist_outside_mm > 0) & (dist_outside_mm <= 5)).astype(np.uint8)

ring1.sum(), ring3.sum(), ring5.sum()


In [ ]:
# kiểm tra trên 1 lát cắt
plt.imshow(ct_clip[z,:,:], cmap="gray")
plt.contour(tumor_mask[z,:,:], colors="r", linewidths=1)
plt.contour(ring1[z,:,:], colors="y", linewidths=1)
plt.title("Red: tumor | Yellow: ring1")
plt.axis("off")
plt.show()


## 4) Tính vài feature đơn giản
Demo: mean HU, median HU, std HU trong intra và ring.

In [ ]:
def basic_features(img, mask, prefix):
    vox = img[mask==1]
    return {
        f"{prefix}_meanHU": float(np.mean(vox)),
        f"{prefix}_medianHU": float(np.median(vox)),
        f"{prefix}_stdHU": float(np.std(vox, ddof=1)),
        f"{prefix}_n_voxels": int(len(vox)),
    }

features = {}
features.update(basic_features(ct_clip, tumor_mask, "intra"))
features.update(basic_features(ct_clip, ring1, "ring1"))
features.update(basic_features(ct_clip, ring3, "ring3"))
features.update(basic_features(ct_clip, ring5, "ring5"))

pd.Series(features).to_frame("value")


## 5) Xuất ra CSV demo
Trong dự án thật, bước này sẽ là trích xuất radiomics bằng PyRadiomics trên nhiều bệnh nhân, rồi ghép lại thành CSV features.

In [ ]:
out = pd.DataFrame([features])
out.to_csv("demo_radiomics_features_one_case.csv", index=False)
out.head()
